# iSnobal Evaluation Report
Run top-to-bottom. Set `CONFIG_PATH` in the Setup cell to point at your config.yml.

## Section 0 — Setup

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, str(pathlib.Path('../scripts').resolve()))
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from eval_utils import *
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

CONFIG_PATH = 'config.yml'
cfg = load_config(CONFIG_PATH)
out_dir = pathlib.Path(cfg['paths']['output_dir'])
out_dir.mkdir(parents=True, exist_ok=True)

snow_ds  = load_snow(cfg)
em_ds    = load_em(cfg)
ns_ds    = load_net_solar(cfg)
topo_ds  = load_topo(cfg)
topo_ds  = compute_aspect(topo_ds)
snotel_df = load_snotel(cfg)

print(f"Basin: {cfg['basin'].upper()}  WY{cfg['water_year']}")
print(f"Snow time steps: {len(snow_ds.time)}")
print(f"SNOTEL sites found: {snotel_df['site_name'].nunique() if not snotel_df.empty else 0}")

## Section 1 — Point Temporal
Single-pixel time series for snow state and energy balance variables.

In [ ]:
x_pt = float(snow_ds.x.values[len(snow_ds.x) // 2])
y_pt = float(snow_ds.y.values[len(snow_ds.y) // 2])
print(f'Pixel: x={x_pt:.0f}, y={y_pt:.0f}')

pt_df = extract_point_timeseries(snow_ds, em_ds, ns_ds, x_pt, y_pt)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Row 1: snow state
ax = axes[0]
for col, label, color in [
    ('thickness',     'Depth (m)',           'steelblue'),
    ('specific_mass', 'SWE (kg/m²) ÷ 500',   'navy'),
    ('snow_density',  'Density (kg/m³) ÷ 500','darkcyan'),
]:
    if col in pt_df.columns:
        vals = pt_df[col] / 500 if col != 'thickness' else pt_df[col]
        ax.plot(pt_df.index, vals, label=label, color=color, lw=1.2)
ax.set_ylabel('Snow state (scaled)')
ax.legend(fontsize=8, ncol=3)
ax.grid(alpha=0.3)

# Row 2: radiation & turbulent fluxes
ax = axes[1]
flux_vars = [
    ('net_rad',      'Net Rad',     'orangered'),
    ('net_solar',    'Net SW',      'gold'),
    ('net_LW',       'Net LW',      'coral'),
    ('sensible_heat','Sensible',    'purple'),
    ('latent_heat',  'Latent',      'mediumorchid'),
]
for col, label, color in flux_vars:
    if col in pt_df.columns:
        ax.plot(pt_df.index, pt_df[col], label=label, color=color, lw=1.2)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_ylabel('Energy flux (W/m²)')
ax.legend(fontsize=8, ncol=5)
ax.grid(alpha=0.3)

# Row 3: mass fluxes
ax = axes[2]
for col, label, color in [
    ('snowmelt',    'Melt (kg/m²)',   'tomato'),
    ('SWI',         'SWI (kg/m²)',    'darkorange'),
    ('evaporation', 'Evap (kg/m²)',   'teal'),
]:
    if col in pt_df.columns:
        ax.plot(pt_df.index, pt_df[col], label=label, color=color, lw=1.2)
ax.set_ylabel('Mass flux (kg/m²/day)')
ax.legend(fontsize=8, ncol=3)
ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

fig.suptitle(f"{cfg['basin'].upper()} WY{cfg['water_year']} — Point Temporal ({x_pt:.0f}, {y_pt:.0f})", fontsize=12)
fig.tight_layout()
fig.savefig(out_dir / 'point_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved point_timeseries.png')

## Section 2 — Spatial Distribution by Terrain
Mean snow depth in each terrain/veg bin at three representative dates.

In [ ]:
time_indices = [0, 120, 210]  # Oct 1, ~Feb 1, ~May 1
date_labels = [str(snow_ds.time.values[i])[:10] for i in time_indices]
stratify_options = ['elevation', 'aspect', 'slope', 'veg_class']

# Compute distribution for all time steps once per stratification
dist_dfs = {}
for strat in stratify_options:
    if strat == 'aspect' and 'aspect' not in topo_ds:
        continue
    try:
        dist_dfs[strat] = compute_terrain_distribution(snow_ds, 'thickness', topo_ds, stratify_by=strat)
    except Exception as e:
        print(f'{strat}: {e}')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.ravel()
colors = ['steelblue', 'darkorange', 'green']

for ax, strat in zip(axes, stratify_options):
    if strat not in dist_dfs:
        ax.set_visible(False)
        continue
    df = dist_dfs[strat]
    bin_cols = df.columns.tolist()
    x_pos = np.arange(len(bin_cols))
    width = 0.25
    for k, (tidx, dlabel, color) in enumerate(zip(time_indices, date_labels, colors)):
        row = df.iloc[tidx]
        ax.bar(x_pos + k * width, row[bin_cols].values, width=width, label=dlabel, color=color, alpha=0.8)
    ax.set_xticks(x_pos + width)
    ax.set_xticklabels(bin_cols, rotation=45, ha='right', fontsize=7)
    ax.set_title(f'By {strat}', fontsize=10)
    ax.set_ylabel('Mean depth (m)')
    ax.legend(fontsize=7)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle(f"{cfg['basin'].upper()} WY{cfg['water_year']} — Snow Depth by Terrain", fontsize=12)
fig.tight_layout()
fig.savefig(out_dir / 'spatial_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved spatial_distribution.png')

## Section 3 — Conditional Time Series
Basin-mean ± spread for pixels matching the terrain filter in config.yml.

In [ ]:
tf = cfg['terrain_filter']
mask = build_terrain_mask(
    topo_ds,
    elev_range=tf.get('elev_range'),
    aspect_range=tf.get('aspect_range'),
    slope_max=tf.get('slope_max'),
    veg_classes=tf.get('veg_classes'),
)
print(f'Pixels in mask: {int(mask.values.sum())} / {mask.size}')

plot_vars = [
    (snow_ds,  'thickness',     'Snow depth (m)',      'steelblue'),
    (snow_ds,  'specific_mass', 'SWE (kg/m²)',         'navy'),
    (ns_ds,    'net_solar',     'Net solar (W/m²)',    'gold'),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

for ax, (ds, var, ylabel, color) in zip(axes, plot_vars):
    ts_df = compute_masked_timeseries(ds, var, mask)
    ax.plot(ts_df.index, ts_df['mean'], color=color, lw=1.5, label='mean')
    ax.fill_between(ts_df.index, ts_df['q25'], ts_df['q75'], color=color, alpha=0.25, label='IQR')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
elev_str = f"elev {tf.get('elev_range', 'all')} m, aspect {tf.get('aspect_range', 'all')}°"
fig.suptitle(f"{cfg['basin'].upper()} WY{cfg['water_year']} — Conditional TS ({elev_str})", fontsize=11)
fig.tight_layout()
fig.savefig(out_dir / 'conditional_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved conditional_timeseries.png')

## Section 4 — SNOTEL Validation
Model vs. observed comparison at SNOTEL sites within the basin polygon.

In [ ]:
if snotel_df.empty:
    print('No SNOTEL sites found — skipping validation.')
else:
    comp_df = compare_snotel(snow_ds, snotel_df, 'thickness')

    # Metrics table
    metric_rows = []
    for site, grp in comp_df.groupby('site_id'):
        m = compute_metrics(grp['modeled'].values, grp['observed'].values)
        m['site_id'] = site
        metric_rows.append(m)

    metrics_df = pd.DataFrame(metric_rows).set_index('site_id')
    metrics_df = metrics_df[['n', 'r', 'r2', 'rmse', 'mae', 'kge']].round(3)
    print('\nSNOTEL validation metrics (snow depth, m):')
    print(metrics_df.to_string())
    metrics_df.to_csv(out_dir / 'snotel_metrics.csv')
    print('\nSaved snotel_metrics.csv')

    # Time series plots
    sites = comp_df['site_id'].unique()[:6]
    ncols = 2
    nrows = int(np.ceil(len(sites) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows), sharex=False)
    axes = np.array(axes).ravel()

    for ax, site in zip(axes, sites):
        grp = comp_df[comp_df['site_id'] == site].set_index('date').sort_index()
        ax.plot(grp.index, grp['modeled'],  color='steelblue',  lw=1.5, label='iSnobal')
        ax.plot(grp.index, grp['observed'], color='darkorange', lw=1.5, ls='--', label='SNOTEL')
        m = metrics_df.loc[site] if site in metrics_df.index else {}
        kge_str = f"KGE={m.get('kge', float('nan')):.2f}" if hasattr(m, 'get') else ''
        ax.set_title(f'{site}  {kge_str}', fontsize=9)
        ax.set_ylabel('Depth (m)')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

    for ax in axes[len(sites):]:
        ax.set_visible(False)

    fig.suptitle(f"{cfg['basin'].upper()} WY{cfg['water_year']} — SNOTEL Validation", fontsize=12)
    fig.tight_layout()
    fig.savefig(out_dir / 'snotel_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved snotel_comparison.png')

## Section 5 — Monthly Summary Table
Basin-wide daily means grouped by month.

In [ ]:
basin_mask_np = topo_ds['mask'].values.astype(bool) if 'mask' in topo_ds else np.ones(snow_ds.dims['y'] * snow_ds.dims['x'], dtype=bool).reshape(snow_ds.dims['y'], snow_ds.dims['x'])

monthly_rows = []
summary_vars = [
    (snow_ds, 'thickness',     'Depth (m)'),
    (snow_ds, 'specific_mass', 'SWE (kg/m²)'),
    (em_ds,   'snowmelt',      'Melt (kg/m²/d)'),
    (em_ds,   'SWI',           'SWI (kg/m²/d)'),
]

for ds, var, label in summary_vars:
    arr = ds[var].values  # (time, y, x)
    basin_mean = np.nanmean(arr[:, basin_mask_np], axis=1)
    ts = pd.Series(basin_mean, index=pd.to_datetime(ds.time.values), name=label)
    monthly = ts.groupby(ts.index.month).agg(['mean', 'max', 'min']).round(3)
    monthly.index.name = 'month'
    monthly.columns = pd.MultiIndex.from_tuples([(label, c) for c in monthly.columns])
    monthly_rows.append(monthly)

summary = pd.concat(monthly_rows, axis=1)
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
summary.index = summary.index.map(month_names)
print(f"\n{cfg['basin'].upper()} WY{cfg['water_year']} — Monthly Basin Means")
print(summary.to_string())